# Proyecto M2

**Alumno:** Ivan Francisco Ramírez  
**Matrícula:** 20221087  
**Grupo:** A  
**Materia:** Extracción de Conocimiento en Bases de Datos  
**Proyecto:** M2 - Proyecto de Mes: Regresión  
**Dataset:** Body Performance Data  
**Variable objetivo:** `body fat_%`  

En este proyecto se desarrollará un modelo de regresión supervisada utilizando la metodología CRISP-DM. El dataset seleccionado contiene información física y de rendimiento corporal de diferentes personas.

El objetivo principal es predecir el porcentaje de grasa corporal, representado por la variable `body fat_%`, a partir de variables como edad, género, altura, peso, presión arterial, fuerza de agarre, flexibilidad, abdominales y salto largo.

Durante el proyecto se realizarán dos escenarios:

1. Modelos entrenados con características seleccionadas.
2. Modelos entrenados con componentes principales obtenidos mediante PCA.

En ambos escenarios se entrenarán los mismos tres algoritmos de regresión y se compararán sus resultados mediante validación cruzada con siete particiones.


El objetivo del proyecto es entrenar y evaluar modelos de regresión capaces de predecir la variable `body fat_%`.

Esta variable representa el porcentaje de grasa corporal de una persona. Sus valores se expresan en porcentaje, por lo que permiten estimar la proporción del cuerpo que corresponde a grasa.

Predecir esta variable puede ser útil para apoyar evaluaciones físicas, seguimiento de condición corporal, análisis deportivo o recomendaciones generales de acondicionamiento físico.

## 1.3 Posible aplicación del modelo

Un modelo de este tipo podría utilizarse en gimnasios, centros deportivos o programas de evaluación física para estimar el porcentaje de grasa corporal de una persona con base en características corporales y pruebas de rendimiento.

El modelo podría apoyar decisiones relacionadas con seguimiento físico, comparación de progreso, orientación de entrenamiento o análisis de condición corporal.

Sin embargo, una predicción incorrecta podría llevar a interpretaciones equivocadas sobre el estado físico de una persona. Por ello, el modelo debe utilizarse como herramienta de apoyo y no como sustituto de una evaluación profesional.

# 2. Comprensión de los datos

En esta etapa se analiza el dataset antes de realizar transformaciones importantes. El objetivo es conocer su estructura, identificar sus atributos, revisar valores nulos, duplicados, posibles inconsistencias, valores atípicos y distribución de la variable objetivo.


In [ ]:
# Importamos las librerías principales

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# URL remota del dataset en GitHub


url = "https://raw.githubusercontent.com/IvanRamirezFrancisco/proyecto-m2-regresion-/refs/heads/main/bodyPerformance.csv"

# Cargamos el dataset
df = pd.read_csv(url)

# Mostramos las primeras filas
df.head()

In [ ]:
# Revisamos las dimensiones originales del dataset

df.shape

In [ ]:
# Revisamos los nombres de las columnas

df.columns

In [ ]:
# Eliminamos espacios al inicio o final de los nombres de columnas, si existieran

df.columns = df.columns.str.strip()

# Verificamos nuevamente los nombres
df.columns

In [ ]:
# Información general del dataset

df.info()

In [ ]:
# Revisamos valores nulos por columna

df.isnull().sum()

In [ ]:
porcentaje_nulos = (df.isnull().sum() / len(df)) * 100
porcentaje_nulos.sort_values(ascending=False)

In [ ]:
df.duplicated().sum()

In [ ]:
df["body fat_%"].describe()

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(df["body fat_%"], bins=25)

plt.title("Distribución del porcentaje de grasa corporal")
plt.xlabel("Porcentaje de grasa corporal")
plt.ylabel("Cantidad de registros")

plt.show()

La gráfica permite observar la distribución de la variable objetivo `body fat_%`. Esta variable será utilizada como `y` porque representa un valor numérico que se desea predecir. Al tratarse de un porcentaje, corresponde a un problema de regresión.

In [ ]:
resumen_atributos = pd.DataFrame({
    "Atributo": df.columns,
    "Tipo de dato en Python": df.dtypes.values,
    "Nulos": df.isnull().sum().values,
    "Porcentaje de nulos": ((df.isnull().sum() / len(df)) * 100).round(2).values,
    "Valores únicos": [df[col].nunique() for col in df.columns]
})

resumen_atributos

In [ ]:
ejemplos_valores = pd.DataFrame({
    "Columna": df.columns,
    "Ejemplos de valores": [df[col].drop_duplicates().head(5).tolist() for col in df.columns],
    "Cantidad de valores únicos": [df[col].nunique() for col in df.columns]
})

ejemplos_valores

## 2.1 Calidad de los datos

En esta sección se revisan posibles problemas que podrían afectar el entrenamiento de los modelos, como valores nulos, registros duplicados, columnas constantes, columnas con poca variabilidad y valores atípicos.

Estas revisiones permiten decidir si alguna columna debe limpiarse, transformarse o eliminarse antes del modelado.

In [ ]:
# Resumen general de calidad de datos

resumen_calidad = pd.DataFrame({
    "Tipo de dato": df.dtypes,
    "Valores nulos": df.isnull().sum(),
    "Porcentaje de nulos": (df.isnull().mean() * 100).round(2),
    "Valores únicos": df.nunique()
})

resumen_calidad

In [ ]:
# Revisamos registros duplicados

duplicados = df.duplicated().sum()

print("Cantidad de registros duplicados:", duplicados)

In [ ]:
# Revisamos si existen columnas constantes

columnas_constantes = df.columns[df.nunique() == 1]

columnas_constantes

In [ ]:
# Revisamos el porcentaje del valor más frecuente en cada columna

porcentaje_mas_frecuente = df.apply(
    lambda columna: columna.value_counts(normalize=True).iloc[0] * 100
)

tabla_variabilidad = pd.DataFrame({
    "Valores únicos": df.nunique(),
    "Porcentaje del valor más frecuente": porcentaje_mas_frecuente.round(2)
})

tabla_variabilidad.sort_values(
    by="Porcentaje del valor más frecuente",
    ascending=False
)

## 2.2 Estadísticas descriptivas

Las estadísticas descriptivas permiten conocer el comportamiento general de las variables numéricas. Se revisan medidas como media, desviación estándar, valor mínimo, máximo y cuartiles.

In [ ]:
# Estadísticas descriptivas de variables numéricas en formato de tabla

estadisticas_descriptivas = df.describe().T

estadisticas_descriptivas

In [ ]:
# Revisamos registros duplicados

duplicados = df.duplicated().sum()

print("Cantidad de registros duplicados:", duplicados)

## 2.3 Histogramas de variables principales

Los histogramas permiten observar la distribución de las variables numéricas principales. Con ellos se puede identificar si los datos se concentran en ciertos rangos, si existen sesgos o si aparecen valores extremos.

In [ ]:
# Histograma de la variable objetivo body fat_%

plt.figure(figsize=(7, 4))

plt.hist(df["body fat_%"], bins=25)

plt.title("Distribución del porcentaje de grasa corporal")
plt.xlabel("Porcentaje de grasa corporal")
plt.ylabel("Cantidad de registros")
plt.grid(True)

plt.show()

En este histograma se observa la distribución de la variable objetivo `body fat_%`. Esta variable representa el porcentaje de grasa corporal de cada persona. La gráfica permite identificar en qué rangos se concentran más los valores y si existen valores extremos.

Esta variable será utilizada como `y` porque es numérica y representa un valor continuo que se desea predecir.

In [ ]:
# Histograma de la edad

plt.figure(figsize=(7, 4))

plt.hist(df["age"], bins=25)

plt.title("Distribución de la edad")
plt.xlabel("Edad")
plt.ylabel("Cantidad de registros")
plt.grid(True)

plt.show()

El histograma de `age` permite observar la distribución de edades de las personas evaluadas. Esta revisión es importante porque la edad puede influir en la composición corporal y en el porcentaje de grasa corporal.

In [ ]:
# Histograma del peso corporal

plt.figure(figsize=(7, 4))

plt.hist(df["weight_kg"], bins=25)

plt.title("Distribución del peso corporal")
plt.xlabel("Peso en kg")
plt.ylabel("Cantidad de registros")
plt.grid(True)

plt.show()

El histograma de `weight_kg` muestra cómo se distribuye el peso corporal de las personas del dataset. Esta variable es relevante porque el peso puede relacionarse con el porcentaje de grasa corporal, aunque por sí solo no determina completamente la composición corporal.

In [ ]:
# Histograma de la fuerza de agarre

plt.figure(figsize=(7, 4))

plt.hist(df["gripForce"], bins=25)

plt.title("Distribución de la fuerza de agarre")
plt.xlabel("Fuerza de agarre")
plt.ylabel("Cantidad de registros")
plt.grid(True)

plt.show()

El histograma de `gripForce` permite observar la distribución de la fuerza de agarre. Esta variable puede reflejar parte del rendimiento físico de la persona y puede tener relación con la composición corporal.

In [ ]:
# Histograma de la cantidad de abdominales

plt.figure(figsize=(7, 4))

plt.hist(df["sit-ups counts"], bins=25)

plt.title("Distribución de abdominales realizados")
plt.xlabel("Cantidad de abdominales")
plt.ylabel("Cantidad de registros")
plt.grid(True)

plt.show()

El histograma de `sit-ups counts` muestra la distribución de la cantidad de abdominales realizados. Esta variable representa una prueba de rendimiento físico y puede relacionarse con la condición corporal de las personas evaluadas.

## 2.4 Boxplots de variables principales

Los boxplots permiten identificar la mediana, la dispersión y posibles valores atípicos en las variables numéricas. Los puntos fuera de los bigotes pueden representar valores extremos.

En esta etapa los valores atípicos solo se identifican; no se eliminan automáticamente porque pueden representar casos reales.

In [ ]:
# Boxplot del porcentaje de grasa corporal por género

plt.figure(figsize=(8, 5))

df.boxplot(
    column="body fat_%",
    by="gender",
    figsize=(8, 5)
)

plt.title("Porcentaje de grasa corporal por género")
plt.suptitle("")
plt.xlabel("Género")
plt.ylabel("Porcentaje de grasa corporal")
plt.xticks(rotation=0)
plt.grid(True)

plt.show()

In [ ]:
# Boxplot del peso corporal por género

plt.figure(figsize=(8, 5))

df.boxplot(
    column="weight_kg",
    by="gender",
    figsize=(8, 5)
)

plt.title("Peso corporal por género")
plt.suptitle("")
plt.xlabel("Género")
plt.ylabel("Peso en kg")
plt.xticks(rotation=0)
plt.grid(True)

plt.show()

In [ ]:
# Boxplot de fuerza de agarre por género

plt.figure(figsize=(8, 5))

df.boxplot(
    column="gripForce",
    by="gender",
    figsize=(8, 5)
)

plt.title("Fuerza de agarre por género")
plt.suptitle("")
plt.xlabel("Género")
plt.ylabel("Fuerza de agarre")
plt.xticks(rotation=0)
plt.grid(True)

plt.show()

In [ ]:
# Boxplot de abdominales por género

plt.figure(figsize=(8, 5))

df.boxplot(
    column="sit-ups counts",
    by="gender",
    figsize=(8, 5)
)

plt.title("Abdominales realizados por género")
plt.suptitle("")
plt.xlabel("Género")
plt.ylabel("Cantidad de abdominales")
plt.xticks(rotation=0)
plt.grid(True)

plt.show()

In [ ]:
# Boxplot de salto largo por género

plt.figure(figsize=(8, 5))

df.boxplot(
    column="broad jump_cm",
    by="gender",
    figsize=(8, 5)
)

plt.title("Salto largo por género")
plt.suptitle("")
plt.xlabel("Género")
plt.ylabel("Salto largo en cm")
plt.xticks(rotation=0)
plt.grid(True)

plt.show()

## 2.5 Distribución de la variable categórica gender

La variable `gender` es categórica y representa el género de la persona evaluada. Se revisa su distribución para identificar si existe un desbalance importante entre categorías.

Esta revisión es necesaria porque las variables categóricas deben convertirse a una representación numérica antes del modelado.

In [ ]:
# Gráfica de barras para la variable gender

plt.figure(figsize=(7, 4))

df["gender"].value_counts().plot(kind="bar")

plt.title("Distribución de la variable gender")
plt.xlabel("Género")
plt.ylabel("Cantidad de registros")
plt.xticks(rotation=0)
plt.grid(axis="y")

plt.show()

La gráfica de barras permite observar cuántos registros existen por cada categoría de `gender`. Esta revisión ayuda a identificar si existe desbalance entre los grupos.

La variable `gender` se conservará como variable independiente, pero antes del modelado deberá convertirse a valores numéricos mediante una técnica de codificación.

## 2.6 Matriz de correlación

La matriz de correlación permite analizar la relación lineal entre variables numéricas. Los valores cercanos a 1 indican una relación positiva fuerte, los valores cercanos a -1 indican una relación negativa fuerte y los valores cercanos a 0 indican poca relación lineal.

Esta matriz será utilizada como uno de los criterios para la selección de características, aunque no será el único método.

In [ ]:
# Matriz de correlación de variables numéricas principales

variables_correlacion = [
    "body fat_%",
    "age",
    "height_cm",
    "weight_kg",
    "diastolic",
    "systolic",
    "gripForce",
    "sit and bend forward_cm",
    "sit-ups counts",
    "broad jump_cm"
]

corr = df[variables_correlacion].corr()

plt.figure(figsize=(10, 8))

sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1
)

plt.title("Matriz de correlación de variables principales")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()

plt.show()

La matriz de correlación muestra qué variables numéricas tienen mayor relación lineal con `body fat_%`.

Las correlaciones positivas indican que al aumentar una variable también tiende a aumentar el porcentaje de grasa corporal. Las correlaciones negativas indican que al aumentar una variable, el porcentaje de grasa corporal tiende a disminuir.

Esta información ayuda a identificar variables candidatas para el modelado, pero no se debe seleccionar únicamente con correlación, porque este método solo mide relaciones lineales.

## 2.7 Correlación con la variable objetivo

Además de observar la matriz completa, se analiza de forma específica la correlación de las variables numéricas con la variable objetivo `body fat_%`.

Esto permite identificar qué variables tienen mayor relación positiva o negativa con el porcentaje de grasa corporal.

In [ ]:
# Correlación de las variables numéricas con la variable objetivo

correlacion_objetivo = corr["body fat_%"].drop("body fat_%").sort_values()

plt.figure(figsize=(8, 5))

correlacion_objetivo.plot(kind="barh")

plt.title("Correlación con body fat_%")
plt.xlabel("Valor de correlación")
plt.ylabel("Variable")
plt.grid(axis="x")

plt.show()

Esta gráfica permite observar de forma directa qué variables tienen mayor relación con `body fat_%`.

Las variables con correlación positiva tienden a aumentar cuando aumenta el porcentaje de grasa corporal. Las variables con correlación negativa tienden a disminuir cuando aumenta el porcentaje de grasa corporal.

La correlación se utilizará como primer método de selección de características, pero no será suficiente por sí sola. También se aplicarán RFE e importancia mediante árboles.

## 2.8 Diagramas de dispersión entre variables independientes y body fat_%

Los diagramas de dispersión permiten observar la relación visual entre una variable independiente y la variable objetivo `body fat_%`.

Cada punto representa una persona del dataset. Estas gráficas ayudan a identificar posibles relaciones positivas, negativas, patrones no lineales o dispersión.

In [ ]:
# Relación entre peso y porcentaje de grasa corporal

plt.figure(figsize=(7, 5))

plt.scatter(
    df["weight_kg"],
    df["body fat_%"],
    alpha=0.5
)

plt.title("Relación entre peso y porcentaje de grasa corporal")
plt.xlabel("Peso en kg")
plt.ylabel("Porcentaje de grasa corporal")
plt.grid(True)

plt.show()

In [ ]:
# Relación entre altura y porcentaje de grasa corporal

plt.figure(figsize=(7, 5))

plt.scatter(
    df["height_cm"],
    df["body fat_%"],
    alpha=0.5
)

plt.title("Relación entre altura y porcentaje de grasa corporal")
plt.xlabel("Altura en cm")
plt.ylabel("Porcentaje de grasa corporal")
plt.grid(True)

plt.show()

In [ ]:
# Relación entre abdominales y porcentaje de grasa corporal

plt.figure(figsize=(7, 5))

plt.scatter(
    df["sit-ups counts"],
    df["body fat_%"],
    alpha=0.5
)

plt.title("Relación entre abdominales y porcentaje de grasa corporal")
plt.xlabel("Cantidad de abdominales")
plt.ylabel("Porcentaje de grasa corporal")
plt.grid(True)

plt.show()

In [ ]:
# Relación entre salto largo y porcentaje de grasa corporal

plt.figure(figsize=(7, 5))

plt.scatter(
    df["broad jump_cm"],
    df["body fat_%"],
    alpha=0.5
)

plt.title("Relación entre salto largo y porcentaje de grasa corporal")
plt.xlabel("Salto largo en cm")
plt.ylabel("Porcentaje de grasa corporal")
plt.grid(True)

plt.show()

# 3. Preparación de los datos

En esta etapa se preparan los datos para el modelado. Se define la variable dependiente `y`, las variables independientes `X`, se eliminan columnas que no deben utilizarse, se revisan valores faltantes y se preparan las transformaciones para variables numéricas y categóricas.

El preprocesamiento se realizará mediante `Pipeline` y `ColumnTransformer` para evitar fuga de información durante la validación cruzada.

In [ ]:
# Creamos una copia del dataset para la etapa de modelado

df_modelo = df.copy()

df_modelo.head()

In [ ]:
# Eliminamos registros duplicados antes del modelado

df_modelo = df_modelo.drop_duplicates()

# Creamos características derivadas a partir de variables independientes

df_modelo["BMI"] = df_modelo["weight_kg"] / ((df_modelo["height_cm"] / 100) ** 2)
df_modelo["pulse_pressure"] = df_modelo["systolic"] - df_modelo["diastolic"]
df_modelo["grip_per_weight"] = df_modelo["gripForce"] / df_modelo["weight_kg"]
df_modelo["jump_per_height"] = df_modelo["broad jump_cm"] / df_modelo["height_cm"]
df_modelo["situps_per_weight"] = df_modelo["sit-ups counts"] / df_modelo["weight_kg"]

df_modelo[
    [
        "height_cm",
        "weight_kg",
        "BMI",
        "pulse_pressure",
        "grip_per_weight",
        "jump_per_height",
        "situps_per_weight"
    ]
].head()

Se crearon variables derivadas a partir de variables independientes del dataset.

`BMI` resume la relación entre peso y altura. `pulse_pressure` representa la diferencia entre presión sistólica y diastólica. `grip_per_weight`, `jump_per_height` y `situps_per_weight` relacionan pruebas de rendimiento físico con peso o altura.

Estas variables no usan la variable objetivo `body fat_%`, por lo tanto no representan fuga de información.

In [ ]:

variable_objetivo = "body fat_%"


In [ ]:
# Revisamos las columnas disponibles antes de definir X e y

df_modelo.columns

In [ ]:
df_modelo.head()

La columna `class` se conservará como variable categórica porque representa una categoría de rendimiento corporal. Esta variable puede aportar información útil para estimar el porcentaje de grasa corporal.

No se utilizará como variable objetivo, sino como una característica independiente más. Posteriormente será transformada mediante `OneHotEncoder`, igual que la variable `gender`.

In [ ]:
# Definimos variables independientes X y variable dependiente y

X = df_modelo.drop(columns=[variable_objetivo])
y = df_modelo[variable_objetivo]

print("Tamaño de X:", X.shape)
print("Tamaño de y:", y.shape)

`X` contiene las variables independientes o predictoras, es decir, las características que se utilizarán para estimar el porcentaje de grasa corporal.

`y` contiene la variable dependiente `body fat_%`, que es el valor que los modelos intentarán predecir.

In [ ]:
# Identificamos columnas numéricas y categóricas

columnas_numericas = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
columnas_categoricas = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Columnas numéricas:")
print(columnas_numericas)

print("\nColumnas categóricas:")
print(columnas_categoricas)

In [ ]:
# Revisamos valores nulos en las variables independientes

X.isnull().sum()

In [ ]:
# Revisamos valores nulos en la variable dependiente

y.isnull().sum()

## 3.8 Generación de valores faltantes artificiales

Al revisar los datos se identificó que el dataset no contiene valores faltantes en las variables independientes ni en la variable dependiente.

Como el proyecto solicita aplicar tratamiento de valores faltantes, se generarán valores nulos artificiales únicamente en algunas variables independientes. La variable dependiente `body fat_%` no será modificada.

La generación se realizará con una semilla fija para que el proceso sea reproducible.

In [ ]:
X_nulos = X.copy()

In [ ]:
# Seleccionamos aleatoriamente el 3% de los registros para diastolic
indices_diastolic = X_nulos.sample(
    frac=0.03,
    random_state=42
).index

X_nulos.loc[indices_diastolic, "diastolic"] = np.nan

In [ ]:
# Seleccionamos aleatoriamente el 3% de los registros para systolic
indices_systolic = X_nulos.sample(
    frac=0.03,
    random_state=43
).index

X_nulos.loc[indices_systolic, "systolic"] = np.nan


In [ ]:
# Seleccionamos aleatoriamente el 3% de los registros para sit-ups counts
indices_situps = X_nulos.sample(
    frac=0.03,
    random_state=44
).index

X_nulos.loc[indices_situps, "sit-ups counts"] = np.nan

In [ ]:
# Revisamos los valores nulos después de generarlos

X_nulos.isnull().sum()

Se generaron valores faltantes artificiales en tres variables independientes numéricas: `diastolic`, `systolic` y `sit-ups counts`.

Se usó aproximadamente el 3% de los registros en cada columna. La variable dependiente `body fat_%` no fue modificada.

Posteriormente, esos valores faltantes serán tratados mediante imputación dentro del pipeline de preprocesamiento.

## 3.9 Preparación del preprocesamiento

Después de generar valores faltantes artificiales, se prepara el tratamiento de datos que se aplicará antes del entrenamiento de los modelos.

Las variables numéricas serán imputadas y escaladas. La variable categórica `gender` será imputada y convertida a valores numéricos.

Para evitar fuga de información, estas transformaciones se realizarán dentro de un `Pipeline` y un `ColumnTransformer`.

In [ ]:
# Importamos herramientas para imputación, codificación y escalado

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 3.10 Transformación de variables numéricas

Las variables numéricas contienen valores como edad, altura, peso, presión arterial y resultados de pruebas físicas.

Primero se aplicará imputación con la mediana para reemplazar los valores faltantes. Se utiliza la mediana porque es menos sensible a valores extremos que la media.

Después se aplicará `StandardScaler` para estandarizar las variables, ya que tienen escalas diferentes.

In [ ]:
# Transformador para variables numéricas

transformador_numerico = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

## 3.11 Transformación de variable categórica

La variable `gender` es categórica nominal. Esto significa que sus valores representan categorías, pero no tienen un orden numérico natural.

Por esta razón, se utilizará `OneHotEncoder`, ya que permite convertir categorías de texto en columnas numéricas sin asignar un orden artificial.

In [ ]:
# Transformador para variable categórica

transformador_categorico = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

## 3.12 Preprocesador general con ColumnTransformer

`ColumnTransformer` permite aplicar diferentes transformaciones dependiendo del tipo de variable.

A las variables numéricas se les aplicará imputación con mediana y escalado.  
A la variable categórica `gender` se le aplicará imputación con el valor más frecuente y codificación con `OneHotEncoder`.



In [ ]:
# Preprocesador general

preprocesador = ColumnTransformer(
    transformers=[
        ("num", transformador_numerico, columnas_numericas),
        ("cat", transformador_categorico, columnas_categoricas)
    ]
)

## 3.13 Verificación del preprocesamiento

Antes de pasar a la selección de características, se verifica que el preprocesador funcione correctamente.

Esta prueba permite comprobar que los valores faltantes fueron tratados, que la variable categórica fue codificada y que los datos quedaron listos en formato numérico.

In [ ]:
# Aplicamos el preprocesador para verificar que funcione correctamente

X_preprocesado = preprocesador.fit_transform(X_nulos)

X_preprocesado.shape

In [ ]:
# Obtenemos los nombres de las variables después del preprocesamiento

nombres_caracteristicas = preprocesador.get_feature_names_out()

nombres_caracteristicas

In [ ]:
# Convertimos los datos preprocesados en un DataFrame para visualizarlos mejor

X_preprocesado_df = pd.DataFrame(
    X_preprocesado,
    columns=nombres_caracteristicas
)

X_preprocesado_df.head()

Después del preprocesamiento, las variables numéricas quedaron imputadas y escaladas. La variable categórica `gender` fue convertida a columnas numéricas mediante `OneHotEncoder`.

Esto deja los datos en un formato adecuado para continuar con la selección de características, PCA y entrenamiento de modelos de regresión.

In [ ]:
X_preprocesado_df.isnull().sum().sum()

In [ ]:
X_nulos.isnull().sum()

In [ ]:

X_preprocesado.shape

In [ ]:

nombres_caracteristicas

# 4. Selección y transformación de características

En esta etapa se aplican diferentes métodos para identificar las variables más relevantes para predecir `body fat_%`.

Se utilizarán tres métodos:

1. Correlación con la variable objetivo.
2. RFE, Recursive Feature Elimination.
3. Importancia de características mediante árboles.

A partir de estos resultados se seleccionarán entre 4 y 6 características para el primer escenario de modelado.

Además, se aplicará PCA como segundo escenario de transformación de características.

## 4.1 Selección por correlación

La correlación permite medir la relación lineal entre cada característica y la variable objetivo `body fat_%`.

Los valores cercanos a 1 indican una relación positiva fuerte.  
Los valores cercanos a -1 indican una relación negativa fuerte.  
Los valores cercanos a 0 indican poca relación lineal.

Este método se utilizará como primer criterio para identificar variables relevantes.

In [ ]:
# Creamos una copia del conjunto preprocesado para calcular correlaciones

datos_correlacion = X_preprocesado_df.copy()

# Agregamos la variable objetivo
datos_correlacion["body fat_%"] = y.values

# Calculamos la correlación de cada característica con la variable objetivo
correlaciones = datos_correlacion.corr()["body fat_%"].drop("body fat_%")

# Creamos una tabla ordenada por correlación absoluta
tabla_correlacion = pd.DataFrame({
    "Característica": correlaciones.index,
    "Correlación": correlaciones.values,
    "Correlación absoluta": correlaciones.abs().values
})

tabla_correlacion = tabla_correlacion.sort_values(
    by="Correlación absoluta",
    ascending=False
)

tabla_correlacion

In [ ]:
# Gráfica de las características con mayor correlación absoluta

top_correlacion = tabla_correlacion.head(10)

plt.figure(figsize=(8, 5))

plt.barh(
    top_correlacion["Característica"],
    top_correlacion["Correlación absoluta"]
)

plt.title("Características con mayor correlación absoluta con body fat_%")
plt.xlabel("Correlación absoluta")
plt.ylabel("Característica")
plt.gca().invert_yaxis()
plt.grid(axis="x")

plt.show()

La tabla y la gráfica muestran qué características tienen mayor relación lineal con `body fat_%`.

La correlación ayuda a identificar variables importantes, pero no debe utilizarse como único criterio, ya que solo mide relaciones lineales y puede no detectar relaciones más complejas.

## 4.2 Selección con RFE

RFE significa Recursive Feature Elimination. Este método entrena un modelo y elimina de forma progresiva las características menos importantes hasta conservar el número de variables indicado.

En este caso se utilizará `LinearRegression` como modelo base para seleccionar 6 características.


In [ ]:
# Importamos RFE y LinearRegression

from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression 

In [ ]:
# Creamos el modelo base para RFE

modelo_base_rfe = LinearRegression()

# Configuramos RFE para seleccionar 6 características
rfe = RFE(
    estimator=modelo_base_rfe,
    n_features_to_select=6
)

# Entrenamos RFE con los datos preprocesados
rfe.fit(X_preprocesado_df, y)

# Creamos una tabla con los resultados de RFE
tabla_rfe = pd.DataFrame({
    "Característica": X_preprocesado_df.columns,
    "Seleccionada por RFE": rfe.support_,
    "Ranking RFE": rfe.ranking_
})

tabla_rfe = tabla_rfe.sort_values(by="Ranking RFE")

tabla_rfe

## 4.3 Importancia de características mediante árboles

Los modelos basados en árboles pueden calcular la importancia de cada característica al momento de realizar predicciones.

En este proyecto se utilizará `RandomForestRegressor` para identificar qué variables aportan más información para predecir `body fat_%`.

Este método es útil porque puede capturar relaciones no lineales entre las variables.

In [ ]:
# Importamos RandomForestRegressor

from sklearn.ensemble import RandomForestRegressor

In [ ]:
# Creamos y entrenamos un modelo Random Forest para calcular importancia de características

modelo_arbol_importancia = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

modelo_arbol_importancia.fit(X_preprocesado_df, y)

# Creamos tabla de importancia
tabla_importancia = pd.DataFrame({
    "Característica": X_preprocesado_df.columns,
    "Importancia": modelo_arbol_importancia.feature_importances_
})

tabla_importancia = tabla_importancia.sort_values(
    by="Importancia",
    ascending=False
)

tabla_importancia

In [ ]:
# Gráfica de las características más importantes según Random Forest

top_importancia = tabla_importancia.head(10)

plt.figure(figsize=(8, 5))

plt.barh(
    top_importancia["Característica"],
    top_importancia["Importancia"]
)

plt.title("Importancia de características mediante Random Forest")
plt.xlabel("Importancia")
plt.ylabel("Característica")
plt.gca().invert_yaxis()
plt.grid(axis="x")

plt.show()

## 4.4 Selección final de características

Para seleccionar las características finales se combinan los resultados de correlación, RFE e importancia mediante árboles.

El objetivo es seleccionar entre 4 y 6 características que tengan buen comportamiento en los tres métodos.

Como la variable categórica `gender` fue convertida mediante OneHotEncoder, puede aparecer como `cat__gender_F` y `cat__gender_M`. Para la selección final se considera como una sola variable original: `gender`.

In [ ]:
# Agregamos ranking de correlación

tabla_correlacion["Ranking correlación"] = tabla_correlacion["Correlación absoluta"].rank(
    ascending=False,
    method="min"
)

# Agregamos ranking de importancia

tabla_importancia["Ranking importancia"] = tabla_importancia["Importancia"].rank(
    ascending=False,
    method="min"
)

In [ ]:
# Unimos los resultados de los tres métodos

tabla_seleccion = tabla_correlacion.merge(
    tabla_rfe,
    on="Característica",
    how="left"
).merge(
    tabla_importancia,
    on="Característica",
    how="left"
)

# Calculamos un puntaje final
# Menor puntaje significa mejor posición general

tabla_seleccion["Puntaje final"] = (
    tabla_seleccion["Ranking correlación"] +
    tabla_seleccion["Ranking RFE"] +
    tabla_seleccion["Ranking importancia"]
)

tabla_seleccion = tabla_seleccion.sort_values(by="Puntaje final")

tabla_seleccion

In [ ]:
# Convertimos nombres transformados a nombres de variables originales

tabla_seleccion["Variable original"] = (
    tabla_seleccion["Característica"]
    .str.replace("num__", "", regex=False)
    .str.replace("cat__", "", regex=False)
    .str.replace(r"^gender_.*$", "gender", regex=True)
    .str.replace(r"^class_.*$", "class", regex=True)
)

tabla_seleccion[
    [
        "Característica",
        "Variable original",
        "Correlación",
        "Correlación absoluta",
        "Ranking correlación",
        "Ranking RFE",
        "Ranking importancia",
        "Puntaje final"
    ]
]

In [ ]:
# Agrupamos por variable original para evitar repetir variables como gender_F y gender_M

tabla_variables_finales = tabla_seleccion.groupby(
    "Variable original",
    as_index=False
).agg({
    "Puntaje final": "min",
    "Correlación absoluta": "max",
    "Importancia": "max",
    "Ranking RFE": "min"
})

tabla_variables_finales = tabla_variables_finales.sort_values(
    by="Puntaje final"
)

tabla_variables_finales

In [ ]:
# Seleccionamos las 6 mejores variables originales

caracteristicas_seleccionadas = tabla_variables_finales.head(6)["Variable original"].tolist()

caracteristicas_seleccionadas 

Las características seleccionadas serán utilizadas en el primer escenario de modelado.

La selección se realizó considerando los resultados combinados de correlación, RFE e importancia mediante árboles. Esto permite tomar una decisión más completa que usar solo un método.

## 4.5 Análisis de Componentes Principales, PCA

PCA es una técnica de reducción de dimensionalidad. Su objetivo es transformar las variables originales en nuevos componentes principales que concentran la mayor parte de la información del conjunto de datos.

PCA se aplicará como segundo escenario de modelado. Esto permitirá comparar si los modelos funcionan mejor con características seleccionadas o con componentes principales.

PCA debe aplicarse después de imputar, codificar y escalar los datos, porque es sensible a la escala de las variables.

In [ ]:
# Importamos PCA

from sklearn.decomposition import PCA

In [ ]:
# Aplicamos PCA para analizar la varianza explicada

pca_analisis = PCA()

pca_analisis.fit(X_preprocesado_df)

# Obtenemos la varianza explicada por cada componente
varianza_explicada = pca_analisis.explained_variance_ratio_

# Calculamos la varianza acumulada
varianza_acumulada = np.cumsum(varianza_explicada)

# Creamos una tabla con los resultados
tabla_pca = pd.DataFrame({
    "Componente": range(1, len(varianza_explicada) + 1),
    "Varianza explicada": varianza_explicada,
    "Varianza acumulada": varianza_acumulada
})

tabla_pca

In [ ]:
# Gráfica de varianza explicada acumulada por PCA

plt.figure(figsize=(8, 5))

plt.plot(
    tabla_pca["Componente"],
    tabla_pca["Varianza acumulada"],
    marker="o"
)

plt.axhline(y=0.90, linestyle="--", label="90% de varianza")
plt.axhline(y=0.95, linestyle="--", label="95% de varianza")

plt.title("Varianza explicada acumulada por PCA")
plt.xlabel("Número de componentes principales")
plt.ylabel("Varianza explicada acumulada")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
# Número de componentes necesarios para explicar al menos el 95% de la varianza

componentes_95 = np.argmax(varianza_acumulada >= 0.95) + 1

print("Número de características antes de PCA:", X_preprocesado_df.shape[1])
print("Número de componentes para conservar al menos 95% de varianza:", componentes_95)

La gráfica de varianza explicada acumulada permite identificar cuántos componentes principales son necesarios para conservar la mayor parte de la información del dataset.

En este proyecto se utilizará PCA como segundo escenario de modelado. Los modelos se entrenarán comparando:

1. Características seleccionadas.
2. Componentes principales obtenidos mediante PCA.

PCA puede ayudar a reducir dimensionalidad, pero disminuye la interpretabilidad porque los componentes ya no representan directamente variables originales.

En esta etapa se aplicaron tres métodos para seleccionar características: correlación, RFE e importancia mediante árboles.

Con base en los resultados obtenidos, se seleccionaron entre 4 y 6 características para el primer escenario de modelado.

También se aplicó PCA como segundo escenario. El análisis mostró que, de las 11 características preprocesadas, se necesitan 8 componentes principales para conservar al menos el 95% de la varianza.

Por lo tanto, los dos escenarios que se compararán serán:

1. Modelos entrenados con características seleccionadas.
2. Modelos entrenados con componentes principales obtenidos mediante PCA.


# 5. Modelado

En esta etapa se entrenan **tres algoritmos de regresión** en dos escenarios, como solicita el proyecto.

Los modelos seleccionados son:

1. `Ridge Regression`, modelo lineal regularizado.
2. `XGBoost Regressor`, modelo de boosting basado en árboles.
3. `MLPRegressor`, red neuronal multicapa.

Los dos escenarios son:

1. Características seleccionadas.
2. Componentes principales mediante PCA.

Esto genera seis experimentos en total.


In [ ]:

# Importamos modelos y herramientas necesarias

from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.decomposition import PCA
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import time
import joblib

# Si en Google Colab aparece error con xgboost, ejecuta una vez:
# !pip install xgboost

from xgboost import XGBRegressor


In [ ]:

# Verificamos que las características seleccionadas existan en X_nulos

columnas_no_encontradas = [
    col for col in caracteristicas_seleccionadas
    if col not in X_nulos.columns
]

columnas_no_encontradas


In [ ]:

# Creamos el conjunto de datos con las características seleccionadas

X_seleccionadas = X_nulos[caracteristicas_seleccionadas].copy()

X_seleccionadas.head()


In [ ]:

# Identificamos columnas numéricas y categóricas dentro de las características seleccionadas

columnas_numericas_seleccionadas = X_seleccionadas.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

columnas_categoricas_seleccionadas = X_seleccionadas.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Columnas numéricas seleccionadas:")
print(columnas_numericas_seleccionadas)

print("\nColumnas categóricas seleccionadas:")
print(columnas_categoricas_seleccionadas)


In [ ]:

# Creamos el preprocesador para las características seleccionadas

preprocesador_seleccionadas = ColumnTransformer(
    transformers=[
        ("num", transformador_numerico, columnas_numericas_seleccionadas),
        ("cat", transformador_categorico, columnas_categoricas_seleccionadas)
    ]
)

print("Características seleccionadas:")
print(caracteristicas_seleccionadas)

print("\nTamaño de X_seleccionadas:")
print(X_seleccionadas.shape)



En el primer escenario se utilizan únicamente las características seleccionadas mediante correlación, RFE e importancia de árboles.

Además, en Ridge y MLP se agregan interacciones polinómicas de grado 2 dentro del pipeline para capturar relaciones no lineales simples entre las variables seleccionadas. Esto no representa fuga de información porque se aplica dentro del pipeline durante la validación cruzada.


In [ ]:

# En el escenario PCA se utilizan todas las variables independientes

X_pca = X_nulos.copy()

print("Tamaño de X_pca:")
print(X_pca.shape)

print("Número de componentes PCA calculado previamente:")
print(componentes_95)



En el segundo escenario se utilizan todas las variables independientes y después se aplica PCA. PCA permite transformar las variables originales en componentes principales que conservan la mayor parte de la información.


In [ ]:

# Definimos los tres modelos de regresión

modelo_ridge = Ridge(
    alpha=1.0
)

modelo_xgboost = XGBRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

modelo_mlp = MLPRegressor(
    hidden_layer_sizes=(128, 64, 32),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    learning_rate_init=0.001,
    max_iter=1500,
    early_stopping=True,
    random_state=42
)



La red neuronal utilizada tiene tres capas ocultas con 128, 64 y 32 neuronas, es decir, 224 neuronas ocultas en total. La salida tiene una sola neurona porque el objetivo es predecir un valor numérico continuo: `body fat_%`.

`XGBoost Regressor` se agrega porque suele tener buen desempeño en datos tabulares y puede capturar relaciones no lineales.



## 5.1 Pipelines del escenario 1: características seleccionadas


In [ ]:

# Pipeline Ridge con características seleccionadas

pipeline_ridge_seleccionadas = Pipeline(steps=[
    ("preprocesador", preprocesador_seleccionadas),
    ("polinomios", PolynomialFeatures(degree=2, include_bias=False)),
    ("modelo", modelo_ridge)
])


In [ ]:

# Pipeline XGBoost con características seleccionadas

pipeline_xgb_seleccionadas = Pipeline(steps=[
    ("preprocesador", preprocesador_seleccionadas),
    ("modelo", modelo_xgboost)
])


In [ ]:

# Pipeline MLPRegressor con características seleccionadas

pipeline_mlp_seleccionadas = Pipeline(steps=[
    ("preprocesador", preprocesador_seleccionadas),
    ("polinomios", PolynomialFeatures(degree=2, include_bias=False)),
    ("modelo", modelo_mlp)
])



## 5.2 Pipelines del escenario 2: PCA


In [ ]:

# Pipeline Ridge con PCA

pipeline_ridge_pca = Pipeline(steps=[
    ("preprocesador", preprocesador),
    ("pca", PCA(n_components=componentes_95)),
    ("modelo", modelo_ridge)
])


In [ ]:

# Pipeline XGBoost con PCA

pipeline_xgb_pca = Pipeline(steps=[
    ("preprocesador", preprocesador),
    ("pca", PCA(n_components=componentes_95)),
    ("modelo", modelo_xgboost)
])


In [ ]:

# Pipeline MLPRegressor con PCA

pipeline_mlp_pca = Pipeline(steps=[
    ("preprocesador", preprocesador),
    ("pca", PCA(n_components=componentes_95)),
    ("modelo", modelo_mlp)
])



Los pipelines integran preprocesamiento, transformación de características y modelo. Esto ayuda a evitar fuga de información, ya que cada transformación se ajusta dentro de cada partición de validación cruzada.



# 6. Evaluación de los modelos

Se evalúan los seis experimentos mediante validación cruzada K-Fold con 7 particiones. Las métricas utilizadas son `R2` y `MAE`.

Un mejor modelo debe tener mayor `R2` y menor `MAE`.


In [ ]:

# Configuramos K-Fold con 7 particiones

kfold = KFold(
    n_splits=7,
    shuffle=True,
    random_state=42
)


In [ ]:
# Evaluamos Ridge seleccionadas

inicio = time.time()

resultado_ridge_sel = cross_validate(
    pipeline_ridge_seleccionadas,
    X_seleccionadas,
    y,
    cv=kfold,
    scoring={
        "r2": "r2",
        "mae": "neg_mean_absolute_error"
    }
)

tiempo_ridge_sel = time.time() - inicio

r2_ridge_sel = resultado_ridge_sel["test_r2"]
mae_ridge_sel = -resultado_ridge_sel["test_mae"]

print("R2 Ridge seleccionadas:", r2_ridge_sel)
print("Promedio R2 Ridge seleccionadas:", r2_ridge_sel.mean())
print("MAE Ridge seleccionadas:", mae_ridge_sel)
print("Promedio MAE Ridge seleccionadas:", mae_ridge_sel.mean())
print("Tiempo:", tiempo_ridge_sel)


In [ ]:
# Evaluamos XGBoost seleccionadas

inicio = time.time()

resultado_xgb_sel = cross_validate(
    pipeline_xgb_seleccionadas,
    X_seleccionadas,
    y,
    cv=kfold,
    scoring={
        "r2": "r2",
        "mae": "neg_mean_absolute_error"
    }
)

tiempo_xgb_sel = time.time() - inicio

r2_xgb_sel = resultado_xgb_sel["test_r2"]
mae_xgb_sel = -resultado_xgb_sel["test_mae"]

print("R2 XGBoost seleccionadas:", r2_xgb_sel)
print("Promedio R2 XGBoost seleccionadas:", r2_xgb_sel.mean())
print("MAE XGBoost seleccionadas:", mae_xgb_sel)
print("Promedio MAE XGBoost seleccionadas:", mae_xgb_sel.mean())
print("Tiempo:", tiempo_xgb_sel)


In [ ]:
# Evaluamos MLP seleccionadas

inicio = time.time()

resultado_mlp_sel = cross_validate(
    pipeline_mlp_seleccionadas,
    X_seleccionadas,
    y,
    cv=kfold,
    scoring={
        "r2": "r2",
        "mae": "neg_mean_absolute_error"
    }
)

tiempo_mlp_sel = time.time() - inicio

r2_mlp_sel = resultado_mlp_sel["test_r2"]
mae_mlp_sel = -resultado_mlp_sel["test_mae"]

print("R2 MLP seleccionadas:", r2_mlp_sel)
print("Promedio R2 MLP seleccionadas:", r2_mlp_sel.mean())
print("MAE MLP seleccionadas:", mae_mlp_sel)
print("Promedio MAE MLP seleccionadas:", mae_mlp_sel.mean())
print("Tiempo:", tiempo_mlp_sel)


In [ ]:
# Evaluamos Ridge PCA

inicio = time.time()

resultado_ridge_pca = cross_validate(
    pipeline_ridge_pca,
    X_pca,
    y,
    cv=kfold,
    scoring={
        "r2": "r2",
        "mae": "neg_mean_absolute_error"
    }
)

tiempo_ridge_pca = time.time() - inicio

r2_ridge_pca = resultado_ridge_pca["test_r2"]
mae_ridge_pca = -resultado_ridge_pca["test_mae"]

print("R2 Ridge PCA:", r2_ridge_pca)
print("Promedio R2 Ridge PCA:", r2_ridge_pca.mean())
print("MAE Ridge PCA:", mae_ridge_pca)
print("Promedio MAE Ridge PCA:", mae_ridge_pca.mean())
print("Tiempo:", tiempo_ridge_pca)


In [ ]:
# Evaluamos XGBoost PCA

inicio = time.time()

resultado_xgb_pca = cross_validate(
    pipeline_xgb_pca,
    X_pca,
    y,
    cv=kfold,
    scoring={
        "r2": "r2",
        "mae": "neg_mean_absolute_error"
    }
)

tiempo_xgb_pca = time.time() - inicio

r2_xgb_pca = resultado_xgb_pca["test_r2"]
mae_xgb_pca = -resultado_xgb_pca["test_mae"]

print("R2 XGBoost PCA:", r2_xgb_pca)
print("Promedio R2 XGBoost PCA:", r2_xgb_pca.mean())
print("MAE XGBoost PCA:", mae_xgb_pca)
print("Promedio MAE XGBoost PCA:", mae_xgb_pca.mean())
print("Tiempo:", tiempo_xgb_pca)


In [ ]:
# Evaluamos MLP PCA

inicio = time.time()

resultado_mlp_pca = cross_validate(
    pipeline_mlp_pca,
    X_pca,
    y,
    cv=kfold,
    scoring={
        "r2": "r2",
        "mae": "neg_mean_absolute_error"
    }
)

tiempo_mlp_pca = time.time() - inicio

r2_mlp_pca = resultado_mlp_pca["test_r2"]
mae_mlp_pca = -resultado_mlp_pca["test_mae"]

print("R2 MLP PCA:", r2_mlp_pca)
print("Promedio R2 MLP PCA:", r2_mlp_pca.mean())
print("MAE MLP PCA:", mae_mlp_pca)
print("Promedio MAE MLP PCA:", mae_mlp_pca.mean())
print("Tiempo:", tiempo_mlp_pca)


In [ ]:

# Creamos una tabla comparativa con los resultados de los seis experimentos

tabla_resultados = pd.DataFrame({
    "Escenario": [
        "Características seleccionadas",
        "Características seleccionadas",
        "Características seleccionadas",
        "PCA",
        "PCA",
        "PCA"
    ],
    "Modelo": [
        "Ridge Regression",
        "XGBoost Regressor",
        "MLPRegressor",
        "Ridge Regression",
        "XGBoost Regressor",
        "MLPRegressor"
    ],
    "R2 promedio": [
        r2_ridge_sel.mean(),
        r2_xgb_sel.mean(),
        r2_mlp_sel.mean(),
        r2_ridge_pca.mean(),
        r2_xgb_pca.mean(),
        r2_mlp_pca.mean()
    ],
    "Desviación estándar R2": [
        r2_ridge_sel.std(),
        r2_xgb_sel.std(),
        r2_mlp_sel.std(),
        r2_ridge_pca.std(),
        r2_xgb_pca.std(),
        r2_mlp_pca.std()
    ],
    "Varianza R2": [
        r2_ridge_sel.var(),
        r2_xgb_sel.var(),
        r2_mlp_sel.var(),
        r2_ridge_pca.var(),
        r2_xgb_pca.var(),
        r2_mlp_pca.var()
    ],
    "MAE promedio": [
        mae_ridge_sel.mean(),
        mae_xgb_sel.mean(),
        mae_mlp_sel.mean(),
        mae_ridge_pca.mean(),
        mae_xgb_pca.mean(),
        mae_mlp_pca.mean()
    ],
    "Desviación estándar MAE": [
        mae_ridge_sel.std(),
        mae_xgb_sel.std(),
        mae_mlp_sel.std(),
        mae_ridge_pca.std(),
        mae_xgb_pca.std(),
        mae_mlp_pca.std()
    ],
    "Varianza MAE": [
        mae_ridge_sel.var(),
        mae_xgb_sel.var(),
        mae_mlp_sel.var(),
        mae_ridge_pca.var(),
        mae_xgb_pca.var(),
        mae_mlp_pca.var()
    ],
    "Tiempo de evaluación": [
        tiempo_ridge_sel,
        tiempo_xgb_sel,
        tiempo_mlp_sel,
        tiempo_ridge_pca,
        tiempo_xgb_pca,
        tiempo_mlp_pca
    ]
})

tabla_resultados = tabla_resultados.sort_values(
    by=["R2 promedio", "MAE promedio"],
    ascending=[False, True]
)

tabla_resultados



La tabla comparativa permite identificar cuál combinación de escenario y modelo obtiene mejor desempeño. Se considera mejor el modelo con mayor `R2 promedio` y menor `MAE promedio`.


In [ ]:

# Resultados de modelos con características seleccionadas

tabla_resultados[
    tabla_resultados["Escenario"] == "Características seleccionadas"
]


In [ ]:

# Resultados de modelos con PCA

tabla_resultados[
    tabla_resultados["Escenario"] == "PCA"
]


In [ ]:

# Creamos tablas con los resultados por partición

detalle_ridge_sel = pd.DataFrame({
    "Partición": range(1, 8),
    "Experimento": "Ridge - Seleccionadas",
    "R2": r2_ridge_sel,
    "MAE": mae_ridge_sel
})

detalle_xgb_sel = pd.DataFrame({
    "Partición": range(1, 8),
    "Experimento": "XGBoost - Seleccionadas",
    "R2": r2_xgb_sel,
    "MAE": mae_xgb_sel
})

detalle_mlp_sel = pd.DataFrame({
    "Partición": range(1, 8),
    "Experimento": "MLP - Seleccionadas",
    "R2": r2_mlp_sel,
    "MAE": mae_mlp_sel
})

detalle_ridge_pca = pd.DataFrame({
    "Partición": range(1, 8),
    "Experimento": "Ridge - PCA",
    "R2": r2_ridge_pca,
    "MAE": mae_ridge_pca
})

detalle_xgb_pca = pd.DataFrame({
    "Partición": range(1, 8),
    "Experimento": "XGBoost - PCA",
    "R2": r2_xgb_pca,
    "MAE": mae_xgb_pca
})

detalle_mlp_pca = pd.DataFrame({
    "Partición": range(1, 8),
    "Experimento": "MLP - PCA",
    "R2": r2_mlp_pca,
    "MAE": mae_mlp_pca
})

tabla_particiones = pd.concat([
    detalle_ridge_sel,
    detalle_xgb_sel,
    detalle_mlp_sel,
    detalle_ridge_pca,
    detalle_xgb_pca,
    detalle_mlp_pca
])

tabla_particiones


In [ ]:

# Boxplot de R2 de los seis experimentos

plt.figure(figsize=(12, 6))

sns.boxplot(
    data=tabla_particiones,
    x="Experimento",
    y="R2"
)

plt.title("Comparación de R² en los seis experimentos")
plt.xlabel("Experimento")
plt.ylabel("R²")
plt.xticks(rotation=45, ha="right")
plt.grid(True)
plt.tight_layout()

plt.show()



El boxplot permite comparar la estabilidad de los seis experimentos en las siete particiones de validación cruzada. Un modelo estable debe presentar valores altos de `R2` y poca dispersión entre particiones.


In [ ]:

# Seleccionamos el mejor modelo con base en la tabla comparativa

mejor_resultado = tabla_resultados.iloc[0]

mejor_escenario = mejor_resultado["Escenario"]
mejor_modelo = mejor_resultado["Modelo"]
mejor_r2 = mejor_resultado["R2 promedio"]
mejor_mae = mejor_resultado["MAE promedio"]

print("Mejor escenario:", mejor_escenario)
print("Mejor modelo:", mejor_modelo)
print("R2 promedio:", mejor_r2)
print("MAE promedio:", mejor_mae)



## 6.1 Cierre de evaluación

En esta etapa se evaluaron los tres modelos en dos escenarios. Se calcularon `R2`, `MAE`, promedio, desviación estándar, varianza y tiempo de evaluación.

El modelo final se selecciona con base en mayor `R2 promedio` y menor `MAE promedio`.


# 7. Evaluación final del mejor modelo

En esta sección se realiza una evaluación adicional únicamente para el modelo seleccionado como mejor experimento.

Primero se toma el mejor pipeline de acuerdo con la tabla comparativa. Después se realiza una división `train_test_split` con 80 % de entrenamiento y 20 % de prueba. Finalmente, se generan las métricas finales, la gráfica de valores reales contra predichos y la gráfica de residuos.

**Nota:** en esta versión se selecciona manualmente el pipeline final para que el código sea más claro y sencillo de explicar. Si la tabla comparativa muestra otro experimento como ganador, solo se cambian las dos líneas donde se asignan `pipeline_final` y `X_final`.


In [ ]:
# Seleccionamos manualmente el mejor pipeline de acuerdo con la tabla comparativa
# En las pruebas realizadas, el mejor desempeño se obtuvo con MLPRegressor y características seleccionadas.

pipeline_final = pipeline_mlp_seleccionadas
X_final = X_seleccionadas.copy()

nombre_modelo_final = "MLPRegressor"
escenario_final = "Características seleccionadas"

print("Modelo final seleccionado:", nombre_modelo_final)
print("Escenario final seleccionado:", escenario_final)
print("Columnas utilizadas por el modelo final:")
print(X_final.columns.tolist())


## 7.1 Tabla detallada de las siete particiones del mejor modelo

La tabla siguiente muestra los resultados de las siete particiones del mejor experimento seleccionado. También se agregan la media, desviación estándar y varianza de `R2` y `MAE`, como solicita el proyecto.


In [ ]:
# Tabla detallada del mejor modelo
# Si se selecciona otro modelo, cambiar detalle_mlp_sel por el detalle correspondiente.

detalle_mejor_modelo = detalle_mlp_sel.copy()

tabla_mejor_modelo = pd.concat([
    detalle_mejor_modelo[["Partición", "R2", "MAE"]],
    pd.DataFrame({
        "Partición": ["Media", "Desviación estándar", "Varianza"],
        "R2": [
            detalle_mejor_modelo["R2"].mean(),
            detalle_mejor_modelo["R2"].std(),
            detalle_mejor_modelo["R2"].var()
        ],
        "MAE": [
            detalle_mejor_modelo["MAE"].mean(),
            detalle_mejor_modelo["MAE"].std(),
            detalle_mejor_modelo["MAE"].var()
        ]
    })
], ignore_index=True)

tabla_mejor_modelo


La tabla permite revisar la estabilidad del mejor experimento. Un modelo estable debe mantener valores de `R2` similares entre particiones, una desviación estándar reducida y una varianza baja. Si el `R2` medio no alcanza 0.80, se debe explicar que se realizaron intentos de mejora y que las variables del dataset presentan una capacidad limitada para predecir `body fat_%`.


In [ ]:
# Dividimos los datos en entrenamiento y prueba

X_train, X_test, y_train, y_test = train_test_split(
    X_final,
    y,
    test_size=0.20,
    random_state=42
)

print("Tamaño X_train:", X_train.shape)
print("Tamaño X_test:", X_test.shape)
print("Tamaño y_train:", y_train.shape)
print("Tamaño y_test:", y_test.shape)


In [ ]:
# Entrenamos el modelo final con los datos de entrenamiento

pipeline_final.fit(X_train, y_train)

# Generamos predicciones con los datos de prueba
y_pred = pipeline_final.predict(X_test)

# Calculamos métricas finales
r2_final = r2_score(y_test, y_pred)
mae_final = mean_absolute_error(y_test, y_pred)
rmse_final = np.sqrt(mean_squared_error(y_test, y_pred))

print("R2 final:", r2_final)
print("MAE final:", mae_final)
print("RMSE final:", rmse_final)


El valor de `R2` indica qué proporción de la variación de la variable objetivo logra explicar el modelo. El `MAE` indica el error absoluto promedio del modelo y el `RMSE` penaliza más los errores grandes.


In [ ]:
# Gráfica de valores reales vs valores predichos

plt.figure(figsize=(7, 6))

plt.scatter(
    y_test,
    y_pred,
    alpha=0.5
)

min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())

plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    linestyle="--"
)

plt.title("Valores reales vs valores predichos")
plt.xlabel("Valores reales de body fat_%")
plt.ylabel("Valores predichos de body fat_%")
plt.grid(True)

plt.show()


La línea diagonal representa una predicción ideal. Los puntos cercanos a la línea indican mejores predicciones. Si los puntos aparecen dispersos respecto a la línea, significa que el modelo presenta errores mayores en algunos registros.


In [ ]:
# Gráfica de residuos

residuos = y_test - y_pred

plt.figure(figsize=(8, 5))

plt.scatter(
    y_pred,
    residuos,
    alpha=0.5
)

plt.axhline(
    y=0,
    linestyle="--"
)

plt.title("Residuos vs valores predichos")
plt.xlabel("Valores predichos")
plt.ylabel("Residuo = valor real - valor predicho")
plt.grid(True)

plt.show()


La gráfica de residuos permite observar cómo se distribuyen los errores. Lo ideal es que los residuos se mantengan alrededor de cero sin formar un patrón claro. Si se observa un patrón definido, puede indicar que el modelo no está capturando completamente la relación entre las variables predictoras y `body fat_%`.


In [ ]:
# Histograma de residuos

plt.figure(figsize=(8, 5))

plt.hist(
    residuos,
    bins=30
)

plt.title("Distribución de los residuos")
plt.xlabel("Residuo")
plt.ylabel("Frecuencia")
plt.grid(True)

plt.show()


El histograma de residuos permite observar si los errores se concentran cerca de cero. Una concentración alta cerca de cero indica que muchas predicciones tuvieron errores pequeños.


## 7.2 Entrenamiento final y almacenamiento del pipeline

Después de evaluar el modelo final con `train_test_split`, el pipeline se vuelve a entrenar con todos los registros disponibles. Esto se realiza únicamente después de haber seleccionado el mejor experimento, para preparar el modelo que será utilizado en la aplicación de Flask y Render.


In [ ]:
# Entrenamiento final con todos los datos disponibles

pipeline_final.fit(X_final, y)

# Guardamos el pipeline final y las columnas utilizadas por el modelo

joblib.dump(pipeline_final, "modelo_final.pkl")
joblib.dump(X_final.columns.tolist(), "columnas_modelo.pkl")

print("Modelo guardado correctamente.")
print("Archivos generados:")
print("modelo_final.pkl")
print("columnas_modelo.pkl")
print("Columnas utilizadas por el modelo:")
print(X_final.columns.tolist())


Los archivos `modelo_final.pkl` y `columnas_modelo.pkl` deberán copiarse a la carpeta de la aplicación Flask. El archivo del modelo contiene el pipeline completo y el archivo de columnas permite construir correctamente el `DataFrame` de entrada durante el despliegue.


# 8. URL de Render

Cuando la aplicación esté publicada, se deberá colocar aquí la URL pública de Render. Esta misma URL también debe aparecer en el documento PDF y en el archivo `URL_Render.txt`.

**URL de Render:** Pendiente de generar.


# 9. Conclusiones dentro de la libreta

En este proyecto se aplicó la metodología CRISP-DM para resolver un problema de regresión supervisada con el dataset Body Performance Data.

Se realizaron etapas de comprensión de datos, preparación, tratamiento de valores faltantes, transformación de variables categóricas, escalado, selección de características, PCA, modelado y evaluación.

Se compararon tres algoritmos de regresión en dos escenarios, generando seis experimentos. El mejor modelo se seleccionó con base en `R2 promedio`, `MAE promedio`, estabilidad entre particiones y viabilidad para despliegue.

Si el mejor experimento no alcanza un `R2` medio de 0.80, se debe explicar que se realizaron intentos de mejora mediante variables derivadas, inclusión de características categóricas, ajuste de hiperparámetros, comparación de modelos y evaluación de PCA. En ese caso, la principal limitación se relaciona con la capacidad predictiva del dataset para estimar `body fat_%` a partir de las variables disponibles.
